In [29]:
import json
import numpy as np
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import sys
from config import JSON_FILE,DATA_DIR, ensure_dirs

In [18]:
root_folder=str(Path.cwd().parent)
if root_folder not in sys.path:
    sys.path.append(root_folder)

In [23]:
ensure_dirs()

In [24]:
with open(JSON_FILE,'r') as f:
    data = json.load(f)

In [6]:
sequences=[]
labels=[]

In [25]:
for entry in data:
    seq = entry['data']['sequence']
    features = []
    for event in seq:
        hold = event['hold_time']
        flight = max(event['flight_time'], 0)  
        features.append([hold, flight])
    if len(features) >= 10:  
        sequences.append(np.array(features))
        labels.append(entry['label'])  


In [26]:
all_features = np.vstack(sequences)
scaler = StandardScaler()
scaler.fit(all_features)

normalized_sequences = [scaler.transform(s) for s in sequences]

In [27]:
window_size = 100
stride = 50
X = []
y = []

In [28]:

for seq, lbl in zip(normalized_sequences, labels):
    for start in range(0, len(seq) - window_size + 1, stride):
        window = seq[start:start + window_size]
        X.append(window)
        y.append(lbl)

In [30]:
X = np.array(X) 
y = np.array(y)  


np.save(DATA_DIR/ 'X.npy', X)
np.save(DATA_DIR/ 'y.npy', y)

In [31]:
print(f"Processed {len(X)} windows (features shape: {X.shape}, labels shape: {y.shape})")
print(f"Balance: {np.sum(y==1)} positive (you), {np.sum(y==0)} negative (others)")

Processed 170 windows (features shape: (170, 100, 2), labels shape: (170,))
Balance: 96 positive (you), 74 negative (others)
